In [1]:
import os, random
import numpy as np

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)  # only takes effect if set before kernel start; pipeline avoids set-order reliance regardless
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # required for deterministic CuBLAS matmuls
random.seed(SEED)
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    # best-effort determinism; ignore if backend doesn't support it
    try: torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception: pass
except ImportError:
    pass

print("Seeded with", SEED)

Seeded with 42


In [2]:
import json, glob
from datetime import datetime, timezone
from collections import Counter

# wildcard catches all -official* dirs/segments so later windows load too
CADETS_FILES = sorted(glob.glob(
    "../data/darpa/ta1-cadets-e3-official*/ta1-cadets-e3-official*.json*"
))
P = "com.bbn.tc.schema.avro.cdm18."
print(f"Loading {len(CADETS_FILES)} file(s):")
for p in CADETS_FILES:
    print("  ", p)

def utc_ns(y, mo, d, h, mi):
    return int(datetime(y, mo, d, h, mi, tzinfo=timezone.utc).timestamp() * 1e9)

# per-window UTC bounds + verified IOCs + ground truth (times EDT = UTC-4)
# Two-tier GT, both derived from TA5.1 Ground Truth Report E3 (data/TC_Ground_Truth_Report_E3_Update.pdf):
#   gt_campaign   = every technique the report attributes to that window's attack segment
#                   (incl. TTPs not expressible from CADETS host audit, e.g. discovery/priv-esc)
#   gt_observable = subset whose evidence is a CADETS process/file/netflow record AND is in the
#                   pipeline's relation vocabulary -> this is the fair precision/recall target.
# Each technique is annotated with the report action that justifies it.
WINDOWS = {
    "W1": {  # 2018-04-06 ~11:00 EDT, report sec 3.1 (first Nginx exploit; sshd inject + crash)
        "start": utc_ns(2018,4,6,14,0), "end": utc_ns(2018,4,6,17,0),
        "ips": {"81.49.200.166","78.205.235.65","200.36.109.214","139.123.0.113",
                "152.111.159.139","154.143.113.18","61.167.39.128"},
        "paths": {"/tmp/vUgefal","/var/log/devc"},
        # T1190 exploit nginx | T1071 HTTP C2 to OC | T1105 putfile vUgefal/libdrakon/netrecon
        # T1059 elevate+run implants | T1055 inject sshd 809 (failed)
        # T1222 chmod on /tmp/vUgefal during elevate (host audit: EVENT_MODIFY_FILE_ATTRIBUTES)
        # campaign-only: T1046 netrecon nrtcp discovery | T1068 elevate to root
        "gt_observable": {"T1190","T1071","T1105","T1059","T1055","T1222"},
        "gt_campaign":   {"T1190","T1071","T1105","T1059","T1055","T1222","T1046","T1068"},
        # was: T1190,T1059,T1105,T1071,T1222,T1021,T1055 -- dropped T1021 (sshd injection failed,
        # no remote-service login); kept T1222 (host chmod evidence); /var/log/devc write is T1105 staging, NOT T1070
    },
    "W2": {  # 2018-04-11 ~15:00 EDT, report sec 3.8 (Nginx exploit; inject /tmp/grain -> crash)
        "start": utc_ns(2018,4,11,18,30), "end": utc_ns(2018,4,11,19,45),
        "ips": {"25.159.96.207","76.56.184.25","155.162.39.48","198.115.236.119"},
        "paths": {"sendmail","grain","/tmp/grain"},
        # T1190 exploit nginx | T1071 C2 | T1105 putfile sendmail/grain | T1059 shellcode in nginx
        # T1055 inject /tmp/grain 802 (failed)
        "gt_observable": {"T1190","T1071","T1105","T1059","T1055"},
        "gt_campaign":   {"T1190","T1071","T1105","T1059","T1055"},
        # was: T1190,T1059,T1105,T1071 -- added T1055 (report: inject /tmp/grain 802)
    },
    "W3": {  # 2018-04-12 ~14:00 EDT, report sec 3.13 (Nginx exploit; micro apt portscan; cleanup)
        "start": utc_ns(2018,4,12,17,30), "end": utc_ns(2018,4,12,19,0),
        "ips": {"25.159.96.207","76.56.184.25","155.162.39.48","198.115.236.119",
                "53.158.101.118","98.15.44.232","192.113.144.28"},
        "paths": {"tmux-1002","minions","font","XIM","netlog","sendmail","main","/tmp/test"},
        # T1190 exploit nginx | T1071 C2 (drakon + micro apt) | T1105 many putfiles
        # T1059 elevate /tmp/XIM + execfile /tmp/test | T1070.004 eight rm of dropped implants
        # T1222 chmod on font/XIM during elevate (host audit: EVENT_MODIFY_FILE_ATTRIBUTES)
        # campaign-only: T1046 APT>scan port 22 sweep | T1041 recon results over C2 | T1068 elevate
        "gt_observable": {"T1190","T1071","T1105","T1059","T1070.004","T1222"},
        "gt_campaign":   {"T1190","T1071","T1105","T1059","T1070.004","T1222","T1046","T1041","T1068"},
        # was: ...,T1222,...,T1021,T1041 -- kept T1222 (host chmod evidence), reclassified portscan as
        # T1046 discovery (not T1021 remote-services), moved T1041 to campaign-only (weak host evidence)
    },
    "W4": {  # 2018-04-13 ~09:00 EDT, report sec 3.14 (re-exploit; cp implants; inject sshd 20691)
        "start": utc_ns(2018,4,13,12,30), "end": utc_ns(2018,4,13,14,0),
        "ips": {"25.159.96.207","76.56.184.25","155.162.39.48","198.115.236.119","53.158.101.118"},
        "paths": {"pEja72mA","eWq10bVcx","memhelp.so","eraseme","done.so"},
        # T1190 re-exploit nginx | T1071 C2 | T1105 putfile pEja72mA/eWq10bVcx
        # T1059 elevate pEja72mA -> root drakon | T1055 inject memhelp.so/done.so into sshd 20691 (failed)
        # T1222 chmod on /tmp/pEja72mA during elevate (host audit: EVENT_MODIFY_FILE_ATTRIBUTES)
        # campaign-only: T1068 elevate
        "gt_observable": {"T1190","T1071","T1105","T1059","T1055","T1222"},
        "gt_campaign":   {"T1190","T1071","T1105","T1059","T1055","T1222","T1068"},
        # was: ...,T1070.004,... -- dropped T1070.004: sec 3.14 has only cp (no rm/file deletion);
        # added T1222 (host chmod evidence during elevate pEja72mA)
    },
}

# eval target = host-observable subset; keep gt_set as alias so all downstream code is unchanged
for _w in WINDOWS.values():
    _w["gt_set"] = _w["gt_observable"]

GLOBAL_START = min(w["start"] for w in WINDOWS.values())
GLOBAL_END   = max(w["end"]   for w in WINDOWS.values())

def cdm_type(datum):
    return next(iter(datum)).replace(P, "")

# PASS 1: build UUID lookups
subjects, files, netflows = {}, {}, {}
for path in CADETS_FILES:
    with open(path) as f:
        for line in f:
            try: d = json.loads(line)["datum"]
            except: continue
            t = cdm_type(d); obj = d[P + t]; u = obj.get("uuid")
            if t == "Subject":
                props = obj.get("properties", {}).get("map", {})
                subjects[u] = props.get("exec") or obj.get("cmdLine") or "process"
            elif t == "FileObject":
                files[u] = obj.get("type", "FILE")
            elif t == "NetFlowObject":
                netflows[u] = f'{obj.get("remoteAddress")}:{obj.get("remotePort")}'

print("Subjects:", len(subjects), "Files:", len(files), "NetFlows:", len(netflows))

# PASS 2: collect events in any window, tag with window id
def which_window(ts):
    for wid, w in WINDOWS.items():
        if w["start"] <= ts <= w["end"]:
            return wid
    return None

events = []
for path in CADETS_FILES:
    with open(path) as f:
        for line in f:
            try: d = json.loads(line)["datum"]
            except: continue
            if cdm_type(d) != "Event": continue
            e = d[P + "Event"]
            ts = e.get("timestampNanos", 0)
            if not (GLOBAL_START <= ts <= GLOBAL_END): continue
            wid = which_window(ts)
            if wid is None: continue
            e["_window"] = wid
            events.append(e)

events.sort(key=lambda e: (e.get("timestampNanos", 0), e.get("uuid", "")))
per_window = dict(Counter(e["_window"] for e in events))
print("Events per window:", per_window)
print("Total events:", len(events))

# warn if a window matched no events
empty = [wid for wid in WINDOWS if per_window.get(wid, 0) == 0]
if empty:
    print(f"\n!! WARNING: these windows matched 0 events: {empty}")
    print("   -> check file coverage (all -official* dirs loaded?) and UTC time bounds.")


Loading 10 file(s):
   ../data/darpa/ta1-cadets-e3-official-1/ta1-cadets-e3-official-1.json
   ../data/darpa/ta1-cadets-e3-official-1/ta1-cadets-e3-official-1.json.1
   ../data/darpa/ta1-cadets-e3-official-1/ta1-cadets-e3-official-1.json.2
   ../data/darpa/ta1-cadets-e3-official-1/ta1-cadets-e3-official-1.json.3
   ../data/darpa/ta1-cadets-e3-official-1/ta1-cadets-e3-official-1.json.4
   ../data/darpa/ta1-cadets-e3-official-2/ta1-cadets-e3-official-2.json
   ../data/darpa/ta1-cadets-e3-official-2/ta1-cadets-e3-official-2.json.1
   ../data/darpa/ta1-cadets-e3-official/ta1-cadets-e3-official.json
   ../data/darpa/ta1-cadets-e3-official/ta1-cadets-e3-official.json.1
   ../data/darpa/ta1-cadets-e3-official/ta1-cadets-e3-official.json.2
Subjects: 224629 Files: 2305159 NetFlows: 155322
Events per window: {'W1': 376379, 'W2': 174244, 'W3': 318821, 'W4': 212342}
Total events: 1081786


In [3]:
# diagnostic: day/window coverage over all events
from datetime import datetime, timezone

def ns_to_str(ts):
    return datetime.fromtimestamp(ts/1e9, tz=timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

# files matched
print("Files matched by glob:")
for p in CADETS_FILES:
    print("  ", p)
print()

# event time span + per-day histogram
day_hist = Counter()
ev_min, ev_max, n_ev = None, None, 0
for path in CADETS_FILES:
    with open(path) as f:
        for line in f:
            try: d = json.loads(line)["datum"]
            except: continue
            if cdm_type(d) != "Event": continue
            ts = d[P+"Event"].get("timestampNanos", 0)
            if ts <= 0: continue
            n_ev += 1
            ev_min = ts if ev_min is None else min(ev_min, ts)
            ev_max = ts if ev_max is None else max(ev_max, ts)
            day = datetime.fromtimestamp(ts/1e9, tz=timezone.utc).strftime("%Y-%m-%d")
            day_hist[day] += 1

print(f"Total events with a timestamp: {n_ev}")
if ev_min:
    print(f"Event time span: {ns_to_str(ev_min)}  ->  {ns_to_str(ev_max)}\n")
    print("Events per calendar day (UTC):")
    for day in sorted(day_hist):
        print(f"   {day}: {day_hist[day]:>8}")
print()

# does each window overlap the data?
print(f"{'win':>4} {'start (UTC)':>17} {'end (UTC)':>17}  status")
for wid, w in WINDOWS.items():
    in_data = (ev_min is not None) and not (w["end"] < ev_min or w["start"] > ev_max)
    status = "OK: overlaps data" if in_data else "EMPTY: window is outside the data's time span"
    print(f"{wid:>4} {ns_to_str(w['start']):>17} {ns_to_str(w['end']):>17}  {status}")


Files matched by glob:
   ../data/darpa/ta1-cadets-e3-official-1/ta1-cadets-e3-official-1.json
   ../data/darpa/ta1-cadets-e3-official-1/ta1-cadets-e3-official-1.json.1
   ../data/darpa/ta1-cadets-e3-official-1/ta1-cadets-e3-official-1.json.2
   ../data/darpa/ta1-cadets-e3-official-1/ta1-cadets-e3-official-1.json.3
   ../data/darpa/ta1-cadets-e3-official-1/ta1-cadets-e3-official-1.json.4
   ../data/darpa/ta1-cadets-e3-official-2/ta1-cadets-e3-official-2.json
   ../data/darpa/ta1-cadets-e3-official-2/ta1-cadets-e3-official-2.json.1
   ../data/darpa/ta1-cadets-e3-official/ta1-cadets-e3-official.json
   ../data/darpa/ta1-cadets-e3-official/ta1-cadets-e3-official.json.1
   ../data/darpa/ta1-cadets-e3-official/ta1-cadets-e3-official.json.2

Total events with a timestamp: 41350895
Event time span: 2018-04-02 22:07 UTC  ->  2018-04-13 21:35 UTC

Events per calendar day (UTC):
   2018-04-02:   191264
   2018-04-03:  3167212
   2018-04-04:  3291540
   2018-04-05:  3596535
   2018-04-06:  398597

In [4]:
# experiment flags (default off = baseline v2)
# MERGE_DAEMON_FLOWS: fuse a process's file + network activity into one alert
# CARRY_DIRECTION: tag channels inbound/outbound for the T1190/T1203 split
MERGE_DAEMON_FLOWS = False        # separate "nginx loop" experiment; keep off to isolate direction
CARRY_DIRECTION    = True         # tag channels inbound/outbound so Stage 5 can recover T1190

SEMANTIC = {"EVENT_EXECUTE","EVENT_WRITE","EVENT_CREATE_OBJECT","EVENT_FORK",
            "EVENT_MODIFY_FILE_ATTRIBUTES","EVENT_MODIFY_PROCESS","EVENT_UNLINK",
            "EVENT_CHANGE_PRINCIPAL","EVENT_RENAME","EVENT_LINK","EVENT_LOGIN",
            "EVENT_MPROTECT","EVENT_TRUNCATE"}
TRAFFIC  = {"EVENT_CONNECT","EVENT_SENDTO","EVENT_RECVFROM","EVENT_ACCEPT",
            "EVENT_SENDMSG","EVENT_RECVMSG","EVENT_BIND"}
INBOUND  = {"EVENT_ACCEPT","EVENT_BIND"}   # true server-side accept only (RECV* also occurs on outbound channels)

def event_targets(e):
    ips, paths = set(), set()
    for k in ("predicateObject","predicateObject2"):
        ref = e.get(k)
        if ref:
            nf = netflows.get(ref.get(P+"UUID"))
            if nf: ips.add(nf.split(":")[0])
    for k in ("predicateObjectPath","predicateObject2Path"):
        pth = e.get(k)
        if pth:
            s = pth.get("string") if isinstance(pth, dict) else pth
            if s: paths.add(s)
    return ips, paths

def path_matches_ioc(p, iocs):
    # anchored match: exact path or basename equality (no loose substring matching)
    base = p.rsplit("/", 1)[-1]
    for ioc in iocs:
        ioc_base = ioc.rsplit("/", 1)[-1]
        if ioc.startswith("/"):
            if p == ioc or base == ioc_base: return ioc
        elif base == ioc_base:
            return ioc
    return None

def is_mal(wid, ips, paths):
    w = WINDOWS[wid]
    if ips & w["ips"]: return True
    return any(path_matches_ioc(p, w["paths"]) for p in paths)

VERB = {"EVENT_EXECUTE":"executed","EVENT_WRITE":"wrote to","EVENT_CREATE_OBJECT":"created",
        "EVENT_FORK":"forked","EVENT_MODIFY_FILE_ATTRIBUTES":"changed permissions on",
        "EVENT_MODIFY_PROCESS":"modified process","EVENT_UNLINK":"deleted",
        "EVENT_CHANGE_PRINCIPAL":"changed privilege via","EVENT_RENAME":"renamed",
        "EVENT_LINK":"linked","EVENT_LOGIN":"logged in via",
        "EVENT_MPROTECT":"changed memory protection on","EVENT_TRUNCATE":"truncated"}

def role_tag(target):
    # semantic hint for the embedder
    t = str(target).lower()
    if t in ("a socket/pipe", "<unknown>", ""): return ""
    if "/tmp/" in t or t.startswith("tmp"):      return " (file in temp directory)"
    if "/var/log" in t or "log" in t:            return " (file under system log directory)"
    if t.endswith(".so") or "memhelp" in t:      return " (shared library / loadable module)"
    if "sshd" in t or "ssh" in t:                return " (SSH service process)"
    if "nginx" in t:                             return " (web server process)"
    if any(c.isdigit() for c in t) and "." in t and "/" not in t:
        return " (external network host)"
    return ""

def chan_phrase(c):
    if CARRY_DIRECTION and c.get("dir") == "inbound":
        return f'accepted inbound connection from {c["ip"]} ({c["count"]} packets)'
    return f'connected to {c["ip"]} ({c["count"]} packets)'

# classify + collapse, separately per window
alerts = []
for wid in WINDOWS:
    sem, chan = [], {}
    for e in events:
        if e["_window"] != wid: continue
        et = e.get("type","?")
        ips, paths = event_targets(e)
        mal = is_mal(wid, ips, paths)
        subj_ref = e.get("subject") or {}
        eprops = (e.get("properties") or {}).get("map") or {}
        proc = eprops.get("exec") or subjects.get(subj_ref.get(P+"UUID")) or "process"
        if et in TRAFFIC:
            # deterministic IP pick: prefer an IOC IP, else lexicographically smallest,
            # so channel keys/counts never depend on set-iteration (hash) order across runs
            ioc_ips = ips & WINDOWS[wid]["ips"]
            ip = min(ioc_ips) if ioc_ips else (min(ips) if ips else None)
            if ip:
                key = (proc, ip)
                if key not in chan:
                    chan[key] = {"proc":proc,"ip":ip,"count":0,"mal":mal,
                                 "ts":e.get("timestampNanos"),
                                 "dir":"inbound" if et in INBOUND else "outbound"}
                chan[key]["count"] += 1
                # inbound wins (T1190 evidence)
                if et in INBOUND: chan[key]["dir"] = "inbound"
        elif et in SEMANTIC:
            # prefer the IOC-matching path, then any non-linker path, so the rendered
            # target is the payload rather than the dynamic linker (ld-elf.so.1)
            ioc_paths  = sorted(p for p in paths if path_matches_ioc(p, WINDOWS[wid]["paths"]))
            non_linker = sorted(p for p in paths if "ld-elf.so" not in p)
            tgt = (ioc_paths[0] if ioc_paths else
                   non_linker[0] if non_linker else
                   next(iter(sorted(paths)), None) or next(iter(sorted(ips)), None) or "a socket/pipe")
            sem.append({"type":et,"proc":proc,"target":tgt,"mal":mal,"ts":e.get("timestampNanos")})

    mal_sem  = [x for x in sem if x["mal"]]
    ben_sem  = [x for x in sem if not x["mal"]]
    mal_chan = [v for v in chan.values() if v["mal"]]
    ben_chan = [v for v in chan.values() if not v["mal"]]
    n_mal = len(mal_sem) + len(mal_chan)
    ben_keep      = random.sample(ben_sem,  min(len(ben_sem),  max(n_mal*10, 10)))
    ben_chan_keep = random.sample(ben_chan, min(len(ben_chan), max(n_mal, 5)))

    # co-occurrence merge: fuse a process's channels into its semantic alerts
    merged_chan_keys = set()
    if MERGE_DAEMON_FLOWS:
        sem_procs  = {x["proc"] for x in mal_sem} | {x["proc"] for x in ben_keep}
        chan_by_proc = {}
        for c in mal_chan + ben_chan_keep:
            chan_by_proc.setdefault((c["proc"], c["mal"]), []).append(c)

    def emit_sem(x):
        base = f'Process {x["proc"]} {VERB.get(x["type"],x["type"])} {x["target"]}{role_tag(x["target"])}'
        if MERGE_DAEMON_FLOWS:
            cs = chan_by_proc.get((x["proc"], x["mal"]), [])
            if cs:
                joined = "; ".join(chan_phrase(c) for c in cs)
                for c in cs: merged_chan_keys.add((c["proc"], c["ip"], c["mal"]))
                base = f'{base}; also {joined}'
        return base

    for x in mal_sem + ben_keep:
        alerts.append({"window":wid,"label":"malicious" if x["mal"] else "benign","ts":x["ts"],
                       "text":emit_sem(x)})

    for c in mal_chan + ben_chan_keep:
        # skip channels already folded into a semantic alert
        if MERGE_DAEMON_FLOWS and (c["proc"], c["ip"], c["mal"]) in merged_chan_keys:
            continue
        alerts.append({"window":wid,"label":"malicious" if c["mal"] else "benign","ts":c["ts"],
                       "text":f'Process {c["proc"]} {chan_phrase(c)}'})

    print(f"{wid}: {n_mal} malicious ({len(mal_sem)} sem + {len(mal_chan)} chan), "
          f"{len(ben_keep)+len(ben_chan_keep)} benign kept")

alerts.sort(key=lambda a:(a["window"], a["ts"]))
texts  = [a["text"]  for a in alerts]
labels = [a["label"] for a in alerts]
awins  = [a["window"] for a in alerts]
print("\nTotal alerts:", len(alerts))
print("Sample malicious alerts:")
for a in [a for a in alerts if a["label"]=="malicious"][:15]:
    print(f'  [{a["window"]}] {a["text"]}')


W1: 13 malicious (8 sem + 5 chan), 143 benign kept
W2: 6 malicious (3 sem + 3 chan), 66 benign kept
W3: 46 malicious (41 sem + 5 chan), 480 benign kept
W4: 18 malicious (12 sem + 6 chan), 198 benign kept

Total alerts: 970
Sample malicious alerts:
  [W1] Process nginx connected to 78.205.235.65 (302 packets)
  [W1] Process nginx accepted inbound connection from 81.49.200.166 (6 packets)
  [W1] Process nginx wrote to <unknown>
  [W1] Process nginx wrote to <unknown>
  [W1] Process nginx connected to 200.36.109.214 (89 packets)
  [W1] Process nginx wrote to /tmp/vUgefal (file in temp directory)
  [W1] Process nginx changed permissions on /tmp/vUgefal (file in temp directory)
  [W1] Process master executed /tmp/vUgefal (file in temp directory)
  [W1] Process vUgefal connected to 139.123.0.113 (157 packets)
  [W1] Process nginx deleted /tmp/vUgefal (file in temp directory)
  [W1] Process vUgefal connected to 61.167.39.128 (2 packets)
  [W1] Process vUgefal changed permissions on /var/log/d

In [5]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")
emb = model.encode(texts, convert_to_tensor=True, show_progress_bar=True)

THRESHOLD = 0.50
# per-window community detection: communities never span windows (exact per-window attribution)
communities = []
for wid in WINDOWS:
    idx = [j for j, w in enumerate(awins) if w == wid]
    if len(idx) < 3: continue
    sub = util.community_detection(emb[idx], threshold=THRESHOLD, min_community_size=3)
    communities += [[idx[k] for k in c] for c in sub]  # map local -> global indices

print(f"{len(communities)} communities at threshold {THRESHOLD} (per-window)\n")
for i, comm in enumerate(communities):
    cl = [labels[j] for j in comm]
    n_mal = cl.count("malicious")
    purity = max(n_mal, len(comm)-n_mal)/len(comm)
    dom = "MALICIOUS" if n_mal > len(comm)/2 else "benign"
    win_mix = Counter(awins[j] for j in comm).most_common(1)[0][0]
    print(f"C{i}: {len(comm)} alerts | {n_mal} mal | purity {purity:.2f} | {dom} | mainly {win_mix}")
    if dom == "MALICIOUS":
        for j in comm:
            if labels[j]=="malicious":
                print(f"      [{awins[j]}] {texts[j]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/31 [00:00<?, ?it/s]

51 communities at threshold 0.5 (per-window)

C0: 49 alerts | 0 mal | purity 1.00 | benign | mainly W1
C1: 42 alerts | 0 mal | purity 1.00 | benign | mainly W1
C2: 12 alerts | 0 mal | purity 1.00 | benign | mainly W1
C3: 7 alerts | 0 mal | purity 1.00 | benign | mainly W1
C4: 7 alerts | 2 mal | purity 0.71 | benign | mainly W1
C5: 7 alerts | 2 mal | purity 0.71 | benign | mainly W1
C6: 6 alerts | 6 mal | purity 1.00 | MALICIOUS | mainly W1
      [W1] Process vUgefal changed permissions on /var/log/devc (file under system log directory)
      [W1] Process master executed /tmp/vUgefal (file in temp directory)
      [W1] Process vUgefal connected to 61.167.39.128 (2 packets)
      [W1] Process vUgefal connected to 139.123.0.113 (157 packets)
      [W1] Process nginx deleted /tmp/vUgefal (file in temp directory)
      [W1] Process nginx changed permissions on /tmp/vUgefal (file in temp directory)
C7: 5 alerts | 1 mal | purity 0.80 | benign | mainly W1
C8: 4 alerts | 2 mal | purity 0.50 | b

In [6]:
def summarize(threshold, min_size=3):
    comms = []
    for wid in WINDOWS:
        idx = [j for j, w in enumerate(awins) if w == wid]
        if len(idx) < min_size: continue
        sub = util.community_detection(emb[idx], threshold=threshold, min_community_size=min_size)
        comms += [[idx[k] for k in c] for c in sub]
    mal_doms, purities, covered = 0, [], set()
    for comm in comms:
        cl = [labels[j] for j in comm]
        n_mal = cl.count("malicious")
        purities.append(max(n_mal, len(comm)-n_mal)/len(comm))
        if n_mal > len(comm)/2:
            mal_doms += 1
            for j in comm:
                if labels[j]=="malicious": covered.add(j)
    tot = sum(1 for l in labels if l=="malicious")
    ap = sum(purities)/len(purities) if purities else 0
    return len(comms), mal_doms, len(covered), tot, ap

print(f"{'thresh':>7} {'#comm':>6} {'#mal-dom':>9} {'mal-cov':>9} {'avg-purity':>11}")
for t in [0.45,0.50,0.55,0.60,0.65,0.70]:
    n,md_,cov,tot,ap = summarize(t)
    print(f"{t:>7.2f} {n:>6} {md_:>9} {cov:>3}/{tot:<5} {ap:>11.2f}")

 thresh  #comm  #mal-dom   mal-cov  avg-purity
   0.45     54        11  45/83           0.90
   0.50     51         9  48/83           0.93
   0.55     59        12  42/83           0.93
   0.60     64        12  55/83           0.96
   0.65     60        12  51/83           0.97
   0.70     65        11  49/83           0.99


In [7]:
import re
from llama_cpp import Llama, LlamaGrammar

MODEL_PATH = "../models/Mistral-Nemo-Instruct-2407-Q5_K_M.gguf"
llm = Llama(model_path=MODEL_PATH, n_ctx=4096, n_gpu_layers=-1, seed=SEED, verbose=False)
print("Model loaded.")

def normalise_entity(text):
    text = re.sub(r'[^a-z0-9_]+','_', text.lower().strip())
    return re.sub(r'_+','_', text).strip('_')[:80]

SECURITY_RELATIONS = ['PERFORMS_RECONNAISSANCE','PERFORMS_PORT_SCAN','BRUTE_FORCES_CREDENTIAL',
    'ACCESS_CREDENTIALS','EXPLOITS_VULNERABILITY','ESTABLISHES_C2','PERFORMS_BEACONING',
    'CAUSES_DENIAL_OF_SERVICE','MOVES_LATERALLY','EXFILTRATES_DATA','EXECUTES_PAYLOAD']
# drop network-only relations: host audit data can't evidence them
NETWORK_ONLY = {'PERFORMS_RECONNAISSANCE','PERFORMS_PORT_SCAN','PERFORMS_BEACONING',
                'CAUSES_DENIAL_OF_SERVICE','BRUTE_FORCES_CREDENTIAL'}
HOST_RELATIONS = [r for r in SECURITY_RELATIONS if r not in NETWORK_ONLY] + \
    ['CHANGES_PERMISSIONS','DELETES_FILE','INJECTS_PROCESS','WRITES_FILE']

RELATION_TO_TECHNIQUES = {
    'PERFORMS_RECONNAISSANCE':['T1046','T1595','T1590'], 'PERFORMS_PORT_SCAN':['T1046','T1595'],
    'BRUTE_FORCES_CREDENTIAL':['T1110','T1110.001','T1110.003'], 'ACCESS_CREDENTIALS':['T1555','T1078','T1110'],
    'EXPLOITS_VULNERABILITY':['T1190','T1203'], 'ESTABLISHES_C2':['T1071','T1071.001','T1071.004'],
    'PERFORMS_BEACONING':['T1071','T1071.004'], 'CAUSES_DENIAL_OF_SERVICE':['T1498','T1499','T1499.001'],
    'MOVES_LATERALLY':['T1021','T1570'], 'EXFILTRATES_DATA':['T1041','T1048'],
    'EXECUTES_PAYLOAD':['T1059','T1059.007','T1105'],
    'CHANGES_PERMISSIONS':['T1222','T1548'], 'DELETES_FILE':['T1070.004','T1070'],
    'INJECTS_PROCESS':['T1055','T1055.001'], 'WRITES_FILE':['T1105'],
}

TRIPLE_SCHEMA = {'type':'object','properties':{'triples':{'type':'array','minItems':1,'maxItems':4,
    'items':{'type':'object','properties':{'subject':{'type':'string'},
        'relation':{'type':'string','enum':HOST_RELATIONS},'target':{'type':'string'}},
    'required':['subject','relation','target']}}},'required':['triples']}
host_grammar = LlamaGrammar.from_json_schema(json.dumps(TRIPLE_SCHEMA))

RELATION_GUIDE = """Relation definitions (use exactly as written):
  EXPLOITS_VULNERABILITY : initial exploit of a service (malformed request to a web server)
  ESTABLISHES_C2         : outbound connection to an external command-and-control address
  EXECUTES_PAYLOAD       : running a dropped binary or command
  WRITES_FILE            : writing/dropping a file to disk
  CHANGES_PERMISSIONS    : changing file permissions (often to enable execution/elevation)
  INJECTS_PROCESS        : injecting code into another process
  DELETES_FILE           : removing a file (often to cover tracks)
  PERFORMS_PORT_SCAN     : probing multiple ports/hosts on the network
  EXFILTRATES_DATA       : transferring data off the host"""
print("Grammar + bridge ready —", len(HOST_RELATIONS), "relations")

# split EXPLOITS_VULNERABILITY -> T1190 (inbound) vs T1203 (local) by direction
SPLIT_EXPLOIT_RELATION = True
EXPLOIT_INBOUND_TECH = "T1190"   # inbound
EXPLOIT_LOCAL_TECH   = "T1203"   # no inbound evidence


llama_context: n_ctx_seq (4096) < n_ctx_train (1024000) -- the full capacity of the model will not be utilized


Model loaded.
Grammar + bridge ready — 10 relations


In [8]:
def extract_triples(alert_texts, tag):
    block = "\n".join(f"- {t}" for t in alert_texts[:6])
    assert "malicious" not in block and "benign" not in block, f"label leak {tag}"
    prompt = f"""[INST] You are a cybersecurity analyst analyzing host audit events.
Extract 1-4 semantic triples describing the attack behaviour.

{RELATION_GUIDE}

subject = process/actor; relation = the ONE best fit; target = file/address/process.
Name what you observe. No generic placeholders.

Example (illustrative FORMAT ONLY -- do NOT copy these names or relations):
{{"triples": [
  {{"subject":"some_process","relation":"WRITES_FILE","target":"some_file"}},
  {{"subject":"some_process","relation":"CHANGES_PERMISSIONS","target":"some_file"}}
]}}

ALERTS ({tag}):
{block}

Return ONLY valid JSON. [/INST]"""
    out = llm(prompt, max_tokens=512, temperature=0, seed=SEED,
              grammar=host_grammar, repeat_penalty=1.1, stop=["[/INST]"])
    try: triples = json.loads(out["choices"][0]["text"].strip()).get("triples", [])
    except Exception as e:
        print(f"  [{tag}] parse failed: {e}"); triples = []
    valid, seen_fallback = [], 0
    for t in triples:
        s = normalise_entity(str(t.get("subject",""))); r = str(t.get("relation","")).upper().strip()
        o = normalise_entity(str(t.get("target","")))
        if not s or not o: continue
        if r not in HOST_RELATIONS: r = "EXECUTES_PAYLOAD"; seen_fallback += 1
        valid.append({"subject":s,"relation":r,"target":o})
    return valid, seen_fallback

def malicious_dominant(comms):
    return [i for i,c in enumerate(comms)
            if sum(1 for j in c if labels[j]=="malicious") > len(c)/2]

MAL_COMMS = malicious_dominant(communities)
print("Malicious-dominant communities:", MAL_COMMS, "\n")

community_triples, fallback_counts = {}, {}
for cid in MAL_COMMS:
    member_texts = [texts[j] for j in communities[cid]]
    win = Counter(awins[j] for j in communities[cid]).most_common(1)[0][0]
    tr, fb = extract_triples(member_texts, f"C{cid}")
    community_triples[cid] = {"triples":tr, "window":win}
    fallback_counts[cid] = fb
    print(f"C{cid} (mainly {win}):")
    for t in tr:
        print(f"   ({t['subject']}, {t['relation']}, {t['target']})")
    if fb: print(f"   [coerced to fallback: {fb}]")
    print()

Malicious-dominant communities: [6, 15, 24, 25, 29, 32, 33, 37, 48] 

C6 (mainly W1):
   (vugefal, CHANGES_PERMISSIONS, var_log_devc)
   (master, EXECUTES_PAYLOAD, tmp_vugefal)
   (vugefal, ESTABLISHES_C2, 61_167_39_128)
   (nginx, DELETES_FILE, tmp_vugefal)

C15 (mainly W2):
   (nginx, WRITES_FILE, grain)
   (nginx, ESTABLISHES_C2, 155_162_39_48)
   (nginx, ESTABLISHES_C2, 76_56_184_25)

C24 (mainly W3):
   (sshd, CHANGES_PERMISSIONS, var_empty)
   (sshd, EXECUTES_PAYLOAD, tmp_test)
   (sshd, ESTABLISHES_C2, 128_55_12_10)

C25 (mainly W3):
   (xim, DELETES_FILE, tmp_test)
   (xim, DELETES_FILE, tmp_main)
   (xim, DELETES_FILE, tmp_xim)
   (cron, EXECUTES_PAYLOAD, tmp_tmux_1002)

C29 (mainly W3):
   (nginx, WRITES_FILE, font)
   (nginx, WRITES_FILE, tmux_1002)
   (nginx, ESTABLISHES_C2, 155_162_39_48)
   (nginx, DELETES_FILE, tmp_font)

C32 (mainly W3):
   (xim, WRITES_FILE, main)
   (xim, WRITES_FILE, test)
   (xim, WRITES_FILE, netlog)
   (xim, DELETES_FILE, netlog)

C33 (mainly W3):

In [9]:
import numpy as np
from sentence_transformers import util as sutil

def load_attck(path="../data/attck/enterprise-attack.json"):
    bundle = json.load(open(path))
    techs = {}
    for obj in bundle["objects"]:
        if obj.get("type")!="attack-pattern" or obj.get("revoked") or obj.get("x_mitre_deprecated"):
            continue
        tid = next((r["external_id"] for r in obj.get("external_references",[])
                    if r.get("source_name")=="mitre-attack"), None)
        if not tid: continue
        tac = [p["phase_name"] for p in obj.get("kill_chain_phases",[])
               if p.get("kill_chain_name")=="mitre-attack"]
        techs[tid] = {"name":obj.get("name",""), "description":obj.get("description",""),
                      "tactic":tac[0] if tac else ""}
    return techs

attck = load_attck()
tech_ids = list(attck.keys())
tech_texts = [f"{attck[t]['name']}. {attck[t]['description']}" for t in tech_ids]
tech_embs = model.encode(tech_texts, convert_to_tensor=True, show_progress_bar=True)
print(f"Embedded {len(tech_ids)} ATT&CK techniques")

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Embedded 691 ATT&CK techniques


In [10]:
TOP_K = 5
MIN_REL_SUPPORT = 2   # a relation must appear >=this many times (or be the community's dominant relation)
                      # to enter pred_multi -- suppresses one-off LLM hallucinations

def in_gt_set(gt_set, pred):
    # parent-level match (subtechniques count)
    if not pred or pred == "Unknown": return False
    pred_parent = pred.split(".")[0]
    return any(pred_parent == g.split(".")[0] for g in gt_set)

def retrieve(query, k=TOP_K):
    q = model.encode(query, convert_to_tensor=True)
    sims = sutil.pytorch_cos_sim(q, tech_embs)[0].cpu().numpy()
    return [tech_ids[i] for i in np.argsort(-sims)[:k]]

def kg_candidates(triples):
    rels = [t["relation"] for t in triples]
    if not rels: return []
    dom = Counter(rels).most_common(1)[0][0]
    return [t for t in RELATION_TO_TECHNIQUES.get(dom,[]) if t in attck]

def community_has_inbound(cid):
    # inbound (accepted) connection seen; relies on CARRY_DIRECTION
    return any("accepted inbound connection" in texts[j] for j in communities[cid])

results = []
for cid in MAL_COMMS:
    info = community_triples[cid]
    triples, win = info["triples"], info["window"]
    gt_set = WINDOWS[win]["gt_set"]
    alert_text = " ".join(texts[j] for j in communities[cid])
    triple_text = " ".join(f"{t['subject']} {t['relation']} {t['target']}" for t in triples)
    cands = kg_candidates(triples)
    retrieved = retrieve(alert_text[:400] + " " + triple_text)

    dom_rel = Counter(t["relation"] for t in triples).most_common(1)[0][0] if triples else "none"

    # deterministic T1190/T1203 resolution by direction when split is on
    resolved = None
    if (globals().get("SPLIT_EXPLOIT_RELATION", False)
            and dom_rel == "EXPLOITS_VULNERABILITY"):
        resolved = EXPLOIT_INBOUND_TECH if community_has_inbound(cid) else EXPLOIT_LOCAL_TECH

    if resolved is not None:
        pred = resolved
    elif cands:
        q_emb = model.encode(alert_text[:400], convert_to_tensor=True)
        c_embs = tech_embs[[tech_ids.index(t) for t in cands]]
        sims = sutil.pytorch_cos_sim(q_emb, c_embs)[0].cpu().tolist()
        pred = cands[int(np.argmax(sims))]
    else:
        pred = retrieved[0] if retrieved else "Unknown"

    # ablation baselines over the SAME communities, to isolate each component's contribution
    pred_dict = cands[0] if cands else (retrieved[0] if retrieved else "Unknown")  # first listed technique (no embeddings)
    pred_retr = retrieved[0] if retrieved else "Unknown"                            # top-1 retrieval over all ATT&CK
    # multi-label: map each relation to its primary technique, but require minimum support
    # (>=MIN_REL_SUPPORT occurrences, OR the dominant relation so small communities never lose their top signal)
    rel_counts = Counter(t["relation"] for t in triples)
    dom_relation = rel_counts.most_common(1)[0][0] if rel_counts else None
    kept_rels = [rel for rel, c in rel_counts.items() if c >= MIN_REL_SUPPORT or rel == dom_relation]
    pred_multi = sorted({next((x for x in RELATION_TO_TECHNIQUES.get(rel, []) if x in attck), None)
                         for rel in kept_rels} - {None})
    # gated direction-based initial access: an exploited public-facing daemon that accepts an
    # inbound connection from an external host evidences T1190 even if the LLM didn't name it
    if globals().get("SPLIT_EXPLOIT_RELATION", False) and community_has_inbound(cid):
        pred_multi = sorted(set(pred_multi) | {EXPLOIT_INBOUND_TECH})
    results.append({"cid":cid,"window":win,"gt_set":sorted(gt_set),"pred":pred,"dom_rel":dom_rel,
                    "retr_hit":bool(set(retrieved) & gt_set),"parent":in_gt_set(gt_set,pred),
                    "pred_dict":pred_dict,"parent_dict":in_gt_set(gt_set,pred_dict),
                    "pred_retr":pred_retr,"parent_retr":in_gt_set(gt_set,pred_retr),
                    "pred_multi":pred_multi})

print(f"{'C':>3} {'win':>4} {'pred':>9} {'dict':>9} {'retr1':>9} {'dom_rel':>22} {'hit':>5}  gt_set")
for r in results:
    print(f"{r['cid']:>3} {r['window']:>4} {r['pred']:>9} {r['pred_dict']:>9} {r['pred_retr']:>9} "
          f"{r['dom_rel']:>22} {str(r['parent']):>5}  {','.join(r['gt_set'])}")

def parent(t):  return t.split(".")[0] if t and t != "Unknown" else t
def parents(ts): return {parent(t) for t in ts}

# ---------- per-window technique detection: precision / recall / F1 (parent-level) ----------
# P/R measured over the SET of predicted techniques per window vs its gt_set (parent-level).
def window_prf(get_techs, title, gt_key="gt_set"):
    print(f"\n{title}:")
    print(f"{'win':>4} {'P':>5} {'R':>5} {'F1':>5} {'T1190?':>7}  predicted -> ok | spur | miss")
    macro = []
    for wid in WINDOWS:
        wr = [r for r in results if r["window"] == wid]
        G  = parents(WINDOWS[wid][gt_key])
        if not wr:
            print(f"{wid:>4} {'-':>5} {'-':>5} {'-':>5} {'-':>7}  (no malicious-dominant community)")
            macro.append((0.0, 0.0, 0.0)); continue
        P = set()
        for r in wr: P |= parents(get_techs(r))
        tp = P & G
        prec = len(tp)/len(P) if P else 0.0
        rec  = len(tp)/len(G) if G else 0.0
        f1   = 2*prec*rec/(prec+rec) if (prec+rec) else 0.0
        macro.append((prec, rec, f1))
        print(f"{wid:>4} {prec:>5.2f} {rec:>5.2f} {f1:>5.2f} {('yes' if 'T1190' in P else 'NO'):>7}  "
              f"{sorted(P)} -> ok={sorted(tp)} | spur={sorted(P-G)} | miss={sorted(G-P)}")
    mp, mr, mf = (sum(x[i] for x in macro)/len(macro) for i in range(3))
    print(f"Macro-avg over {len(WINDOWS)} windows:  P={mp:.2f}  R={mr:.2f}  F1={mf:.2f}")
    return mp, mr, mf

window_prf(lambda r: [r["pred"]],     "Single-technique per community (dominant relation) [baseline]", "gt_observable")
window_prf(lambda r: r["pred_multi"], "Multi-label per community (all extracted relations)", "gt_observable")
# secondary view: recall against full campaign GT (includes non-host TTPs like discovery/priv-esc the
# host audit can't evidence) -- precision here is not meaningful, read the recall column only
window_prf(lambda r: r["pred_multi"], "Multi-label vs CAMPAIGN GT (coverage incl. non-host TTPs; read recall only)", "gt_campaign")

# ---------- coverage (recall over the attack itself, not just per-community accuracy) ----------
mal_in_comm = {j for cid in MAL_COMMS for j in communities[cid] if labels[j] == "malicious"}
detected = sum(1 for wid in WINDOWS if any(r["window"] == wid for r in results))
t1190_win = sum(1 for wid in WINDOWS
                if any("T1190" in parents(r["pred_multi"]) for r in results if r["window"] == wid))
print("\nCoverage:")
print(f"  windows with >=1 malicious-dominant community: {detected}/{len(WINDOWS)}")
for wid in WINDOWS:
    tot = sum(1 for j in range(len(awins)) if awins[j] == wid and labels[j] == "malicious")
    cap = sum(1 for j in mal_in_comm if awins[j] == wid)
    print(f"    {wid}: malicious-alert capture {cap}/{tot}")
print(f"  windows where T1190 (initial access) recovered: {t1190_win}/{len(WINDOWS)}")
inbound_comms = sum(1 for cid in MAL_COMMS if community_has_inbound(cid))
print(f"  malicious communities with inbound-accept evidence: {inbound_comms}/{len(MAL_COMMS)}")

# ---------- per-community accuracy + component ablation ----------
n      = max(len(results), 1)
exact  = sum(r["pred"] in r["gt_set"] for r in results)
hits   = sum(r["parent"] for r in results)
hd     = sum(r["parent_dict"] for r in results)
hr     = sum(r["parent_retr"] for r in results)
print(f"\nPer-community accuracy ({len(results)} malicious-dominant communities):")
print(f"  exact-match in gt_set                 : {exact}/{len(results)} = {exact/n:.1%}")
print(f"  parent-match (KG-anchored + emb rank) : {hits}/{len(results)} = {hits/n:.1%}")
print(f"  parent-match dictionary-only          : {hd}/{len(results)} = {hd/n:.1%}")
print(f"  parent-match retrieval-only (top-1)   : {hr}/{len(results)} = {hr/n:.1%}")


  C  win      pred      dict     retr1                dom_rel   hit  gt_set
  6   W1     T1222     T1222 T1574.010    CHANGES_PERMISSIONS  True  T1055,T1059,T1071,T1105,T1190,T1222
 15   W2 T1071.001     T1071 T1036.008         ESTABLISHES_C2  True  T1055,T1059,T1071,T1105,T1190
 24   W3     T1222     T1222 T1548.006    CHANGES_PERMISSIONS  True  T1059,T1070.004,T1071,T1105,T1190,T1222
 25   W3 T1070.004 T1070.004 T1070.004           DELETES_FILE  True  T1059,T1070.004,T1071,T1105,T1190,T1222
 29   W3     T1105     T1105 T1055.005            WRITES_FILE  True  T1059,T1070.004,T1071,T1105,T1190,T1222
 32   W3     T1105     T1105 T1070.002            WRITES_FILE  True  T1059,T1070.004,T1071,T1105,T1190,T1222
 33   W3     T1222     T1222 T1070.002    CHANGES_PERMISSIONS  True  T1059,T1070.004,T1071,T1105,T1190,T1222
 37   W3     T1105     T1105 T1222.002            WRITES_FILE  True  T1059,T1070.004,T1071,T1105,T1190,T1222
 48   W4     T1105     T1105     T1011            WRITES_FILE  Tru

In [11]:
import torch
state = {
    "seed": SEED, "threshold": THRESHOLD,
    "alerts": alerts,
    "communities": [[int(j) for j in c] for c in communities],
    "mal_comms": MAL_COMMS,
    "community_triples": {str(k):v for k,v in community_triples.items()},
    "fallback_counts": {str(k):v for k,v in fallback_counts.items()},
    "results": results,
}
with open("../data/darpa/cadets_v2_run.json","w") as f:
    json.dump(state, f, indent=2)
torch.save(emb, "../data/darpa/cadets_v2_emb.pt")
print("Saved run state + embeddings.")

Saved run state + embeddings.
